# Nifty 50 Sector Rotation & Anomaly Detection
## Notebook 2 — Analysis
**Author:** Prathmesh Joshi  
**Analyses:**
- Daily returns & 30-day rolling metrics
- Volatility ranking per stock
- Sector rotation by quarter
- Anomaly detection using 2-standard-deviation threshold
- Volume spike detection

In [ ]:
import pandas as pd
import numpy as np

print('Libraries loaded')

In [ ]:
# Load raw data saved from Notebook 1
df = pd.read_csv('../data/nifty_raw.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print('Rows loaded:', len(df))
print('Tickers:', df['Ticker'].nunique())
df.head()

## Step 1 — Daily Returns & Rolling Metrics

In [ ]:
# Calculate daily percentage return for each stock
# pct_change() computes (today - yesterday) / yesterday
df['Daily_Return'] = df.groupby('Ticker')['Close'].pct_change()

# 30-day rolling average return — smooths out daily noise
df['Rolling_30D_Return'] = df.groupby('Ticker')['Daily_Return'].transform(
    lambda x: x.rolling(window=30, min_periods=15).mean()
)

# 30-day rolling volatility — standard deviation of daily returns
# Higher value = more volatile / risky stock
df['Volatility_30D'] = df.groupby('Ticker')['Daily_Return'].transform(
    lambda x: x.rolling(window=30, min_periods=15).std()
)

print('Daily return range:', round(df['Daily_Return'].min()*100, 2), '% to', round(df['Daily_Return'].max()*100, 2), '%')
print('Sample:')
df[['Date','Ticker','Sector','Close','Daily_Return','Rolling_30D_Return','Volatility_30D']].dropna().head(10)

## Step 2 — Anomaly Detection

In [ ]:
# Anomaly = day where price moved more than 2 standard deviations
# from its 30-day rolling mean — statistically unusual event
df['Mean_30D'] = df.groupby('Ticker')['Daily_Return'].transform(
    lambda x: x.rolling(window=30, min_periods=15).mean()
)
df['Std_30D'] = df.groupby('Ticker')['Daily_Return'].transform(
    lambda x: x.rolling(window=30, min_periods=15).std()
)

# Flag anomalies — both positive spikes and negative crashes
df['Upper_Band'] = df['Mean_30D'] + 2 * df['Std_30D']
df['Lower_Band'] = df['Mean_30D'] - 2 * df['Std_30D']
df['Anomaly'] = (
    (df['Daily_Return'] > df['Upper_Band']) |
    (df['Daily_Return'] < df['Lower_Band'])
)
df['Anomaly_Direction'] = np.where(
    df['Daily_Return'] > df['Upper_Band'], 'Positive Spike',
    np.where(df['Daily_Return'] < df['Lower_Band'], 'Negative Crash', 'Normal')
)

anomalies = df[df['Anomaly'] == True]
print('Total anomaly events detected:', len(anomalies))
print('\nAnomalies by sector:')
print(anomalies.groupby('Sector').size().sort_values(ascending=False))

In [ ]:
# Top 20 most extreme anomaly events
top_anomalies = anomalies.copy()
top_anomalies['Return_Pct'] = round(top_anomalies['Daily_Return'] * 100, 2)
top_anomalies['Abs_Return'] = top_anomalies['Return_Pct'].abs()
top_anomalies = top_anomalies.sort_values('Abs_Return', ascending=False)

print('Top 20 most extreme anomaly events:')
top_anomalies[['Date','Ticker','Sector','Return_Pct','Volume','Anomaly_Direction']].head(20)

## Step 3 — Volatility Ranking

In [ ]:
# Rank all 15 stocks by average 30-day volatility over the full 3-year period
# This tells us which stocks carry the most risk
volatility_rank = (
    df.groupby(['Ticker', 'Sector'])['Volatility_30D']
    .mean()
    .reset_index()
    .rename(columns={'Volatility_30D': 'Avg_Volatility'})
    .sort_values('Avg_Volatility', ascending=False)
    .reset_index(drop=True)
)
volatility_rank['Rank'] = volatility_rank.index + 1
volatility_rank['Avg_Volatility'] = round(volatility_rank['Avg_Volatility'] * 100, 4)

print('Volatility Ranking (highest to lowest risk):')
print(volatility_rank.to_string(index=False))

## Step 4 — Sector Rotation by Quarter

In [ ]:
# Sector rotation = which sector outperformed each quarter
# Sum daily returns per sector per quarter to get quarterly performance
df['Quarter'] = df['Date'].dt.to_period('Q').astype(str)

sector_rotation = (
    df.groupby(['Sector', 'Quarter'])['Daily_Return']
    .sum()
    .reset_index()
    .rename(columns={'Daily_Return': 'Quarterly_Return'})
)
sector_rotation['Quarterly_Return_Pct'] = round(sector_rotation['Quarterly_Return'] * 100, 2)

# Rank sectors within each quarter
sector_rotation['Rank'] = sector_rotation.groupby('Quarter')['Quarterly_Return'].rank(
    ascending=False, method='dense'
).astype(int)

# Show which sector led each quarter
leaders = sector_rotation[sector_rotation['Rank'] == 1][['Quarter','Sector','Quarterly_Return_Pct']]
print('Sector leading each quarter:')
print(leaders.to_string(index=False))

In [ ]:
# Count how many quarters each sector led — sector dominance summary
print('\nQuarters led by each sector (out of 12 total):')
print(leaders['Sector'].value_counts().to_string())

## Step 5 — Monthly Returns (for Power BI heatmap)

In [ ]:
# Aggregate daily returns to monthly for the Power BI heatmap visual
df['Month'] = df['Date'].dt.to_period('M').astype(str)

monthly = (
    df.groupby(['Ticker', 'Sector', 'Month'])['Daily_Return']
    .sum()
    .reset_index()
    .rename(columns={'Daily_Return': 'Monthly_Return'})
)
monthly['Monthly_Return_Pct'] = round(monthly['Monthly_Return'] * 100, 2)

print('Monthly returns sample:')
monthly.head(10)

## Step 6 — Save All Output Files

In [ ]:
# Save processed files for Power BI and MySQL
df.to_csv('../data/nifty_processed.csv', index=False)
monthly.to_csv('../data/nifty_monthly.csv', index=False)
anomalies.to_csv('../data/nifty_anomalies.csv', index=False)
volatility_rank.to_csv('../data/nifty_volatility.csv', index=False)
sector_rotation.to_csv('../data/nifty_sector_rotation.csv', index=False)

print('All files saved to ../data/')
print('Files:')
print('  nifty_processed.csv    — full dataset with all metrics')
print('  nifty_monthly.csv      — monthly returns per stock')
print('  nifty_anomalies.csv    — anomaly events only')
print('  nifty_volatility.csv   — volatility ranking')
print('  nifty_sector_rotation.csv — quarterly sector performance')

## Key Findings Summary
*(Fill this in after running the analysis — replace the placeholders below)*

In [ ]:
print('=== KEY FINDINGS ===')
print(f"Total anomaly events detected: {len(anomalies)}")
print(f"Most volatile stock: {volatility_rank.iloc[0]['Ticker']}")
print(f"Least volatile stock: {volatility_rank.iloc[-1]['Ticker']}")
print(f"Sector that led most quarters: {leaders['Sector'].value_counts().index[0]}")
print(f"Sector with most anomalies: {anomalies.groupby('Sector').size().idxmax()}")